# Notebook 03 — Production Status and Lead-Time Prediction

Builds two baselines from simulated production records: (1) production-status classification at a production snapshot and (2) lead-time regression for completed orders. Simulated-data results are demonstrations only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, r2_score
RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
paths = [Path('/content/drive/MyDrive/LL JADO/LL_MACHINERY_AI_PROJECT'), Path('/content/drive/MyDrive/LL_MACHINERY_AI_PROJECT')]
PROJECT_DIR = next((p for p in paths if p.exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Project folder not found in Google Drive.')
DATA_PATH = PROJECT_DIR / '01_Data' / 'simulated' / 'SIMULATED_production_records.csv'
MODEL_DIR, EVAL_DIR = PROJECT_DIR / '03_Models', PROJECT_DIR / '04_Evaluation'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
print('Data shape:', df.shape)
display(df.head())
display(df.isna().sum().to_frame('missing_count'))

## Part A — Production status classification

Prediction point: a production snapshot. `elapsed_days` is included because it is known at that snapshot. Actual completion dates and actual lead time are excluded to avoid future-information leakage.

In [ ]:
status_features = ['machine_type', 'model_series', 'quantity', 'n_custom_options', 'open_orders_at_order_date', 'planned_lead_time_days', 'elapsed_days']
status_features = [c for c in status_features if c in df.columns]
status_data = df[status_features + ['production_status']].dropna(subset=['production_status']).copy()
X, y = status_data[status_features], status_data['production_status'].astype(str)
print('Features:', status_features)
display(y.value_counts().to_frame('records'))
if y.nunique() < 2:
    raise ValueError('At least two production status classes are required.')
stratify = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify)
cat = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num = [c for c in X_train.columns if c not in cat]
preprocess = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
status_model = Pipeline([('preprocess', preprocess), ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
status_model.fit(X_train, y_train)
pred = status_model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 4))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45, cmap='Blues')
plt.title('Production Status — Confusion Matrix'); plt.tight_layout(); plt.show()
joblib.dump({'model': status_model, 'feature_columns': status_features, 'target': 'production_status', 'data_source': 'SIMULATED', 'prediction_point': 'production snapshot'}, MODEL_DIR / 'production_status_model.joblib')
pd.DataFrame([{'Model':'Random Forest Classifier','Accuracy':accuracy_score(y_test,pred),'Balanced_Accuracy':balanced_accuracy_score(y_test,pred)}]).to_csv(EVAL_DIR / 'production_status_model_metrics.csv', index=False)

## Part B — Actual lead-time regression

Estimate days from order placement to completion. Train only on completed orders and use information available at order placement. Actual completion dates and actual lead time are never model inputs.

In [ ]:
lead_features = ['machine_type', 'model_series', 'quantity', 'n_custom_options', 'open_orders_at_order_date', 'planned_lead_time_days']
lead_features = [c for c in lead_features if c in df.columns]
completed = df[df['actual_lead_time_days'].notna()].copy()
completed = completed[completed['production_status'].astype(str).str.lower() == 'completed']
print('Completed orders available:', len(completed))
if len(completed) < 10:
    print('Not enough completed orders for a baseline regression (at least 10 required).')
else:
    Xl = completed[lead_features]
    yl = pd.to_numeric(completed['actual_lead_time_days'], errors='coerce')
    valid = yl.notna(); Xl, yl = Xl.loc[valid], yl.loc[valid]
    Xtr, Xte, ytr, yte = train_test_split(Xl, yl, test_size=0.2, random_state=RANDOM_STATE)
    catl = Xtr.select_dtypes(include=['object', 'category']).columns.tolist(); numl = [c for c in Xtr.columns if c not in catl]
    prepl = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numl), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), catl)])
    lead_model = Pipeline([('preprocess', prepl), ('model', RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1))])
    lead_model.fit(Xtr, ytr); predl = lead_model.predict(Xte)
    metrics = {'Model':'Random Forest','MAE_days':mean_absolute_error(yte,predl),'RMSE_days':mean_squared_error(yte,predl)**0.5,'R2':r2_score(yte,predl)}
    display(pd.DataFrame([metrics]).round(3))
    plt.figure(figsize=(6,5)); plt.scatter(yte,predl,alpha=0.7)
    lo=min(yte.min(),predl.min()); hi=max(yte.max(),predl.max()); plt.plot([lo,hi],[lo,hi],'--')
    plt.xlabel('Actual lead time (days)'); plt.ylabel('Predicted lead time (days)'); plt.title('Actual vs Predicted Lead Time'); plt.tight_layout(); plt.show()
    joblib.dump({'model':lead_model,'feature_columns':lead_features,'target':'actual_lead_time_days','data_source':'SIMULATED'}, MODEL_DIR / 'lead_time_model.joblib')
    pd.DataFrame([metrics]).to_csv(EVAL_DIR / 'lead_time_model_metrics.csv', index=False)
print('Status model and evaluation files saved. All results use SIMULATED data; validate with approved company data before operational use.')

## Before using real company data

Confirm when status is recorded and which fields are available at prediction time. Obtain permission, remove customer identifiers, and keep confidential company records out of a public GitHub repository. Inspect class-level precision/recall and the confusion matrix, not accuracy alone.